In [1]:
import scanpy as sc
import treedata as td
from pathlib import Path
import matplotlib.pyplot as plt
from harmonypy import run_harmony

from Moslin.utility import compute_hvg_pca_fastRNA

In [ ]:
# TODO: Annotate the cardiac cell type per time point
# TODO: Run Moslin again and see if there is proper linkage of those cells
# TODO: Make the Moslin code a Function

In [41]:
grouped_markers = {
    "Primitive_streak": ["Nanog", "Eomes", "T"],
    "Nascent_mesoderm": ["Mesp2", "Lefty2", "Tdgf1", "Mesp1"],
    "Mixed_mesoderm": ["Phlda2", "Msx1", "Msx2", "Ifitm1"],
    "Paraxial_Pharyngeal_mesoderm": ["Tcf15"],
    "ExE_mesoderm": ["Cdx1", "Cdx2", "Cdx4", "Bmp4"],
    "aSH_Pharyngeal_mesoderm": ["Isl1", "Fgf10", "Tbx1"],
    "pSHF_Pharyngeal_mesoderm": ["Osr1", "Aldh1a2", "Arg1"],
    "Mesenchyme": ["Igf2", "Krt8", "Krt18", "Pmp22", "Ahnak"],
    "JCF_Mesenchyme": ["Hand1", "Foxf1", "Mab21l2"],
    "Cardiomyocytes": ["Ttn", "Tnnt2", "Myl7", "Acta2"],
}

In [24]:
Lineage_data = Path("/project/imoskowitz/kdreyer/public_datasets/005_Colgan2026/")
target_stages = ["E7.5", "E8.0", "E8.5"]
stage_tdatas = {stage: {} for stage in target_stages}

for file_path in Lineage_data.glob("*.h5td"):
    stem = file_path.stem
    stage = stem.split("-")[0]

    if stage in target_stages:
        stage_tdatas[stage][stem] = td.read_h5td(file_path)

In [ ]:
import seaborn as sns

output_dir = Path("/project/imoskowitz/yubin/Lineage_Tree_Construction/output_plot/Feature_plots")
output_dir.mkdir(parents=True, exist_ok=True)

output_data_dir = "/project/imoskowitz/yubin/Lineage_Tree_Construction/output_data/"
for stage_name, stage_files in stage_tdatas.items():
    if not stage_files:
        print(f"{stage_name}: no files found")
        continue
    merged_tdata = td.concat(stage_files, merge='unique')

    merged_tdata.layers['Raw_count'] = merged_tdata.X.copy()
    merged_tdata = compute_hvg_pca_fastRNA(merged_tdata, batch_key='embryo')

    # Harmony integration on the PCA embedding
    harmony_result = run_harmony(
        merged_tdata.obsm['X_pca'],
        merged_tdata.obs[['embryo']],
        vars_use=['embryo'],
        max_iter_harmony=20,
        theta=2.0,
        nclust=None,
        tau=0,
    )
    merged_tdata.obsm['X_pca_harmony'] = harmony_result.Z_corr

    sc.pp.neighbors(merged_tdata, n_pcs=10, use_rep='X_pca_harmony')
    sc.tl.umap(merged_tdata)

    # Log-transform expression for feature plotting
    sc.pp.normalize_total(merged_tdata)
    sc.pp.log1p(merged_tdata)
    merged_tdata.layers['Norm_count'] = merged_tdata.X.copy()

    print(f"\n=== {stage_name} ===")
    print("cells:", merged_tdata.n_obs)
    print("genes:", merged_tdata.n_vars)
    print(merged_tdata)
    merged_tdata.write_h5td(output_data_dir+stage_name.replace(".","_"))
    print("Saved "+ stage_name)

    


2026-06-30 15:15:46,033 - harmonypy - INFO - Running Harmony
2026-06-30 15:15:46,035 - harmonypy - INFO -   Parameters:
2026-06-30 15:15:46,036 - harmonypy - INFO -     max_iter_harmony: 20
2026-06-30 15:15:46,037 - harmonypy - INFO -     max_iter_kmeans: 4
2026-06-30 15:15:46,038 - harmonypy - INFO -     epsilon_cluster: 0.001
2026-06-30 15:15:46,039 - harmonypy - INFO -     epsilon_harmony: 0.01
2026-06-30 15:15:46,039 - harmonypy - INFO -     nclust: 100
2026-06-30 15:15:46,040 - harmonypy - INFO -     block_size: 0.05
2026-06-30 15:15:46,041 - harmonypy - INFO -     lamb: dynamic (alpha=0.2)
2026-06-30 15:15:46,042 - harmonypy - INFO -     theta: [2. 2. 2.]
2026-06-30 15:15:46,044 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-06-30 15:15:46,045 - harmonypy - INFO -     verbose: True
2026-06-30 15:15:46,045 - harmonypy - INFO -     random_state: 0
2026-06-30 15:15:46,046 - harmonypy - INFO -   Data: 50 PCs × 4648 cells
2026-06-30 15:15:46,047 - harmonypy - INFO -   B


=== E7.5 ===
cells: 4648
genes: 78258
TreeData object with n_obs × n_vars = 4648 × 78258
    obs: 'cell_subtype', 'capture', 'embryo', 'stage', 'type', 'total_counts', 'pe_counts', 'detection_rate', 'edit_frac', 'clone', 'phase', 'cell_type', 'germ_layer', 'lineage', 'tree'
    var: 'gene_ids', 'mito', 'chromosome'
    uns: 'pca', 'neighbors', 'umap', 'log1p'
    obsm: 'X_scvi', 'X_umap', 'characters', 'X_pca', 'X_pca_harmony'
    varm: 'PCs'
    layers: 'Raw_count', 'Norm_count'
    obsp: 'distances', 'connectivities'
    obst: 'E7.5-R2-C6', 'E7.5-R2-C3', 'E7.5-R2-C1', 'E7.5-R3-C1', 'E7.5-R2-C4', 'E7.5-R2-C2', 'E7.5-R3-C2', 'E7.5-R1-C2', 'E7.5-R2-C5', 'E7.5-R1-C1', 'E7.5-R3-C3'
Saved E7.5


2026-06-30 15:16:03,638 - harmonypy - INFO - Running Harmony
2026-06-30 15:16:03,639 - harmonypy - INFO -   Parameters:
2026-06-30 15:16:03,640 - harmonypy - INFO -     max_iter_harmony: 20
2026-06-30 15:16:03,640 - harmonypy - INFO -     max_iter_kmeans: 4
2026-06-30 15:16:03,641 - harmonypy - INFO -     epsilon_cluster: 0.001
2026-06-30 15:16:03,641 - harmonypy - INFO -     epsilon_harmony: 0.01
2026-06-30 15:16:03,642 - harmonypy - INFO -     nclust: 100
2026-06-30 15:16:03,642 - harmonypy - INFO -     block_size: 0.05
2026-06-30 15:16:03,642 - harmonypy - INFO -     lamb: dynamic (alpha=0.2)
2026-06-30 15:16:03,643 - harmonypy - INFO -     theta: [2. 2. 2.]
2026-06-30 15:16:03,644 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-06-30 15:16:03,644 - harmonypy - INFO -     verbose: True
2026-06-30 15:16:03,645 - harmonypy - INFO -     random_state: 0
2026-06-30 15:16:03,645 - harmonypy - INFO -   Data: 50 PCs × 21942 cells
2026-06-30 15:16:03,646 - harmonypy - INFO -   


=== E8.0 ===
cells: 21942
genes: 78258
TreeData object with n_obs × n_vars = 21942 × 78258
    obs: 'cell_subtype', 'capture', 'embryo', 'stage', 'type', 'total_counts', 'pe_counts', 'detection_rate', 'edit_frac', 'clone', 'phase', 'cell_type', 'germ_layer', 'lineage', 'tree'
    var: 'gene_ids', 'mito', 'chromosome'
    uns: 'pca', 'neighbors', 'umap', 'log1p'
    obsm: 'X_scvi', 'X_umap', 'characters', 'X_pca', 'X_pca_harmony'
    varm: 'PCs'
    layers: 'Raw_count', 'Norm_count'
    obsp: 'distances', 'connectivities'
    obst: 'E8.0-R3-C2', 'E8.0-R2-C2', 'E8.0-R1-C2', 'E8.0-R1-C3', 'E8.0-R2-C1', 'E8.0-R3-C5', 'E8.0-R3-C1', 'E8.0-R1-C1', 'E8.0-R3-C6', 'E8.0-R3-C4', 'E8.0-R3-C3'
Saved E8.0


2026-06-30 15:17:03,768 - harmonypy - INFO - Running Harmony
2026-06-30 15:17:03,769 - harmonypy - INFO -   Parameters:
2026-06-30 15:17:03,769 - harmonypy - INFO -     max_iter_harmony: 20
2026-06-30 15:17:03,770 - harmonypy - INFO -     max_iter_kmeans: 4
2026-06-30 15:17:03,770 - harmonypy - INFO -     epsilon_cluster: 0.001
2026-06-30 15:17:03,770 - harmonypy - INFO -     epsilon_harmony: 0.01
2026-06-30 15:17:03,771 - harmonypy - INFO -     nclust: 100
2026-06-30 15:17:03,771 - harmonypy - INFO -     block_size: 0.05
2026-06-30 15:17:03,771 - harmonypy - INFO -     lamb: dynamic (alpha=0.2)
2026-06-30 15:17:03,772 - harmonypy - INFO -     theta: [2. 2. 2.]
2026-06-30 15:17:03,772 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-06-30 15:17:03,773 - harmonypy - INFO -     verbose: True
2026-06-30 15:17:03,773 - harmonypy - INFO -     random_state: 0
2026-06-30 15:17:03,773 - harmonypy - INFO -   Data: 50 PCs × 89230 cells
2026-06-30 15:17:03,773 - harmonypy - INFO -   


=== E8.5 ===
cells: 89230
genes: 78258
TreeData object with n_obs × n_vars = 89230 × 78258
    obs: 'cell_subtype', 'capture', 'embryo', 'stage', 'type', 'total_counts', 'pe_counts', 'detection_rate', 'edit_frac', 'clone', 'phase', 'cell_type', 'germ_layer', 'lineage', 'tree'
    var: 'gene_ids', 'mito', 'chromosome'
    uns: 'pca', 'neighbors', 'umap', 'log1p'
    obsm: 'X_scvi', 'X_umap', 'characters', 'X_pca', 'X_pca_harmony'
    varm: 'PCs'
    layers: 'Raw_count', 'Norm_count'
    obsp: 'distances', 'connectivities'
    obst: 'E8.5-R3-C1', 'E8.5-R2-C1', 'E8.5-R2-C2', 'E8.5-R1-C1', 'E8.5-R1-C3', 'E8.5-R1-C2', 'E8.5-R3-C3', 'E8.5-R3-C2', 'E8.5-R1-C4'
Saved E8.5


#### Plotting Feature Plots

In [20]:
# some_file.py
import sys
# caution: path[0] is reserved for script path (or '' in REPL)
sys.path.insert(1, '/project/imoskowitz/yubin/SmoNull_NMPs_mesoderm_biased_analysis')
from src.I_preprocessing.plot_preprocessing import plot_UMAP_custom
from src.III_celltype_annotation.plot_celltype_annotation import (
    filter_marker_genes,
    get_marker_vmin_vmax,
    plot_marker_genes_feature,
    plot_markers_violin_comparison,
)

In [37]:
merged_tdatas = {}
path = Path("/project/imoskowitz/yubin/Lineage_Tree_Construction/output_data/Processed_data")
for file in path.glob("*"):
    merged_tdatas[file.stem] = td.read_h5td(file)

In [38]:
merged_tdatas.values()

dict_values([TreeData object with n_obs × n_vars = 89230 × 78258
    obs: 'cell_subtype', 'capture', 'embryo', 'stage', 'type', 'total_counts', 'pe_counts', 'detection_rate', 'edit_frac', 'clone', 'phase', 'cell_type', 'germ_layer', 'lineage', 'tree'
    var: 'gene_ids', 'mito', 'chromosome'
    uns: 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_scvi', 'X_umap', 'characters'
    varm: 'PCs'
    layers: 'Norm_count', 'Raw_count'
    obsp: 'connectivities', 'distances'
    obst: 'E8.5-R3-C1', 'E8.5-R2-C1', 'E8.5-R2-C2', 'E8.5-R1-C1', 'E8.5-R1-C3', 'E8.5-R1-C2', 'E8.5-R3-C3', 'E8.5-R3-C2', 'E8.5-R1-C4', TreeData object with n_obs × n_vars = 21942 × 78258
    obs: 'cell_subtype', 'capture', 'embryo', 'stage', 'type', 'total_counts', 'pe_counts', 'detection_rate', 'edit_frac', 'clone', 'phase', 'cell_type', 'germ_layer', 'lineage', 'tree'
    var: 'gene_ids', 'mito', 'chromosome'
    uns: 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X

In [43]:
output_path_plot = "/project/imoskowitz/yubin/Lineage_Tree_Construction/output_plot/"
for day, adata in merged_tdatas.items():
    adata.X = adata.layers["Norm_count"]
    markers_filtered = filter_marker_genes(adata, grouped_markers)
    markers_vmin_vmax = get_marker_vmin_vmax(adata=adata, markers_filtered=markers_filtered)


    plot_marker_genes_feature(
    adata=adata, umap_coords_obsm="X_umap",
    markers_filtered=markers_filtered,
    output_path_plot=output_path_plot+day+"/",
    markers_vmin_vmax=markers_vmin_vmax)

    

    

### Compute Leiden Clustering


In [52]:
from src.II_integration_clustering.plot_integration_clustering import plot_clustering

In [53]:

for day, adata in merged_tdatas.items():    
    output_path_plot_int = output_path_plot + "leiden/"+day
    res_list = [0.25, 0.5, 1.0, 2.0]

    leiden_keys = []
    for res in res_list:
        key = f"leiden_{res}"
        print(f"clustering for resolution {res}...")
        sc.tl.leiden(
            adata, resolution=res,
            key_added=key,
            neighbors_key="neighbors",
            flavor="igraph",
            n_iterations=2,
            directed=False
        )
        leiden_keys.append(key)

    # build filename-safe names without decimals for saving plots
    # e.g. leiden_0.25 -> l025
    leiden_names = [
        "l" + key.split("_")[1].replace(".", "")
        for key in leiden_keys
    ]
    adata.uns["leiden_names"] = leiden_names

    plot_clustering(adata, leiden_keys, output_path_plot_int, umap_coords_obsm = "X_umap")

clustering for resolution 0.25...
clustering for resolution 0.5...
clustering for resolution 1.0...
clustering for resolution 2.0...
clustering for resolution 0.25...
clustering for resolution 0.5...
clustering for resolution 1.0...
clustering for resolution 2.0...
clustering for resolution 0.25...
clustering for resolution 0.5...
clustering for resolution 1.0...
clustering for resolution 2.0...


In [56]:
for day, adata in merged_tdatas.items():
    adata.write_h5td(output_data_dir+day)
    print("Saved "+ day)

Saved E8_5
Saved E8_0
Saved E7_5


In [57]:
adata

TreeData object with n_obs × n_vars = 4648 × 78258
    obs: 'cell_subtype', 'capture', 'embryo', 'stage', 'type', 'total_counts', 'pe_counts', 'detection_rate', 'edit_frac', 'clone', 'phase', 'cell_type', 'germ_layer', 'lineage', 'tree', 'leiden_0.25', 'leiden_0.5', 'leiden_1.0', 'leiden_2.0'
    var: 'gene_ids', 'mito', 'chromosome'
    uns: 'log1p', 'neighbors', 'pca', 'umap', 'leiden_0.25', 'leiden_0.5', 'leiden_1.0', 'leiden_2.0', 'leiden_names', 'leiden_0.25_colors', 'leiden_0.5_colors', 'leiden_1.0_colors', 'leiden_2.0_colors'
    obsm: 'X_pca', 'X_pca_harmony', 'X_scvi', 'X_umap', 'characters'
    varm: 'PCs'
    layers: 'Norm_count', 'Raw_count'
    obsp: 'connectivities', 'distances'
    obst: 'E7.5-R2-C6', 'E7.5-R2-C3', 'E7.5-R2-C1', 'E7.5-R3-C1', 'E7.5-R2-C4', 'E7.5-R2-C2', 'E7.5-R3-C2', 'E7.5-R1-C2', 'E7.5-R2-C5', 'E7.5-R1-C1', 'E7.5-R3-C3'